# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/home/bernard/Projects/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
import numpy as np
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

:: loading settings :: url = jar:file:/home/bernard/Projects/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/bernard/.ivy2.5.2/cache
The jars for the packages stored in: /home/bernard/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e96514d1-4f26-4198-9d7e-e54b33c2aed5;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#d

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='conuty_trip_flow', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='daily_county_demand', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_part', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips_partitioned', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemp

# Ensure uncached tables

In [3]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

# MVT for partition pruning
Let's base our MVT around the default values and on a log scale

In [4]:
spark.sql("""
    SELECT COUNT(*) AS rows
    FROM default.integrated_taxi_trips
    WHERE pu_county IS nulL
""").show()

26/09/18 17:21:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-----+
| rows|
+-----+
|12265|
+-----+



In [13]:
# partition
taxi_trips_part = (
    spark.table("default.taxi_trips")
        .withColumn("pu_date", F.to_date("pu_datetime"))
)
taxi_trips_part.write.mode("overwrite").format("delta").partitionBy("pu_date").saveAsTable("default.taxi_trips_part")

26/09/18 17:35:35 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`default`.`taxi_trips_part` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [14]:
# verify partition
spark.sql("""
    DESCRIBE default.taxi_trips_part
""").show(truncate=False)

+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|pu_datetime            |timestamp|NULL   |
|do_datetime            |timestamp|NULL   |
|pu_location_id         |int      |NULL   |
|do_location_id         |int      |NULL   |
|fare_amount            |float    |NULL   |
|trip_distance          |float    |NULL   |
|pu_date                |date     |NULL   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|pu_date                |date     |NULL   |
+-----------------------+---------+-------+



In [28]:
import time

for i, q_fn in enumerate([query_2_1, query_2_2, query_2_3, query_2_4, query_2_5, query_2_6]):
    q = q_fn()
    
    print(f"\n--- Query {i+1} ---")
    for table in ["taxi_trips", "taxi_trips_part"]:
        sql_to_run = q.replace("taxi_trips", table)
        
        start_time = time.time()

        for _ in range(5):
            # force spark to compute entire dataset 
            spark.sql(sql_to_run).write.format("noop").mode("overwrite").save()
        
        duration = time.time() - start_time
        
        print(f"[{table:<15}] Total Execution Time: {duration/5:.2f}s (avg)")



# sample run

#--- Query 1 ---
#[taxi_trips     ] Total Execution Time: 1.64s (avg)
#[taxi_trips_part] Total Execution Time: 1.87s (avg)

#--- Query 2 ---
#[taxi_trips     ] Total Execution Time: 2.95s (avg)
#[taxi_trips_part] Total Execution Time: 1.62s (avg)

#--- Query 3 ---
#[taxi_trips     ] Total Execution Time: 3.11s (avg)
#[taxi_trips_part] Total Execution Time: 2.73s (avg)

#--- Query 4 ---
#[taxi_trips     ] Total Execution Time: 1.90s (avg)
#[taxi_trips_part] Total Execution Time: 1.65s (avg)

#--- Query 5 ---
#[taxi_trips     ] Total Execution Time: 1.32s (avg)
#[taxi_trips_part] Total Execution Time: 1.20s (avg)

#--- Query 6 ---
#[taxi_trips     ] Total Execution Time: 1.15s (avg)
#[taxi_trips_part] Total Execution Time: 1.02s (avg)


--- Query 1 ---


[taxi_trips     ] Total Execution Time: 1.64s (avg)


[taxi_trips_part] Total Execution Time: 1.87s (avg)

--- Query 2 ---


[taxi_trips     ] Total Execution Time: 2.95s (avg)


[taxi_trips_part] Total Execution Time: 1.62s (avg)

--- Query 3 ---


...

[taxi_trips_part] Total Execution Time: 1.65s (avg)

--- Query 5 ---


[taxi_trips     ] Total Execution Time: 1.32s (avg)


[taxi_trips_part] Total Execution Time: 1.20s (avg)

--- Query 6 ---


[taxi_trips     ] Total Execution Time: 1.15s (avg)


[taxi_trips_part] Total Execution Time: 1.02s (avg)


In [10]:
partition_bytes = np.array([int(0.5*134217728), 1*134217728, 2*134217728, 4*134217728]) # 64MB, 128MB, 256MB, 512MB
partitions = np.array([100, 200, 400, 800])

# get all combinations of partition_bytes and partitions
from itertools import product
combinations = list(product(partition_bytes, partitions))
print(f'combinations: {combinations}')

for partition_bytes, partitions in combinations:
    print(f'Running with partition_bytes={partition_bytes}, partitions={partitions}')
    spark.conf.set("spark.sql.files.maxPartitionBytes", partition_bytes)
    # spark.conf.set("spark.sql.files.openCostInBytes", partition_bytes)
    # spark.conf.set("spark.sql.files.maxPartitionBytes", partition_bytes)
    # spark.conf.set("spark.sql.files.openCostInBytes", partition_bytes)

    spark.conf.set("spark.sql.shuffle.partitions", partitions)
    for i in range(5):
        print(f'Run {i+1} for partition_bytes={partition_bytes}, partitions={partitions}')
        with log_step(f'Run {i+1} for partition_bytes={partition_bytes}, partitions={partitions}'):
            spark.sql(query_2_1()).show()

combinations: [(np.int64(67108864), np.int64(100)), (np.int64(67108864), np.int64(200)), (np.int64(67108864), np.int64(400)), (np.int64(67108864), np.int64(800)), (np.int64(134217728), np.int64(100)), (np.int64(134217728), np.int64(200)), (np.int64(134217728), np.int64(400)), (np.int64(134217728), np.int64(800)), (np.int64(268435456), np.int64(100)), (np.int64(268435456), np.int64(200)), (np.int64(268435456), np.int64(400)), (np.int64(268435456), np.int64(800)), (np.int64(536870912), np.int64(100)), (np.int64(536870912), np.int64(200)), (np.int64(536870912), np.int64(400)), (np.int64(536870912), np.int64(800))]
Running with partition_bytes=67108864, partitions=100
Run 1 for partition_bytes=67108864, partitions=100


+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|        Baisley Park|    1|      999|
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|   Hillcrest/Pomonok|    1|      187|
|       Rockaway Park|    1|       80|
|     Oakland Gardens|    1|       56|
|           Stapleton|    1|        4|
|            Flushing|    1|      250|
|          Ozone Park|    1|      105|
|          Ocean Hill|    1|      354|
|             Maspeth|    1|      290|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|  Brooklyn Navy Yard|    1|       62|
|          Highbridge|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
+--------------------+-----+---------+
only showing top 20 rows
Run 2 for partition_bytes=67108864, par

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|        Baisley Park|    1|      999|
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|   Hillcrest/Pomonok|    1|      187|
|       Rockaway Park|    1|       80|
|     Oakland Gardens|    1|       56|
|           Stapleton|    1|        4|
|            Flushing|    1|      250|
|          Ozone Park|    1|      105|
|          Ocean Hill|    1|      354|
|             Maspeth|    1|      290|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|  Brooklyn Navy Yard|    1|       62|
|          Highbridge|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
+--------------------+-----+---------+
only showing top 20 rows
Run 3 for partition_bytes=67108864, par

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|        Baisley Park|    1|      999|
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|   Hillcrest/Pomonok|    1|      187|
|       Rockaway Park|    1|       80|
|     Oakland Gardens|    1|       56|
|           Stapleton|    1|        4|
|            Flushing|    1|      250|
|          Ozone Park|    1|      105|
|          Ocean Hill|    1|      354|
|             Maspeth|    1|      290|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|  Brooklyn Navy Yard|    1|       62|
|          Highbridge|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
+--------------------+-----+---------+
only showing top 20 rows
Run 4 for partition_bytes=67108864, par

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|        Baisley Park|    1|      999|
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|   Hillcrest/Pomonok|    1|      187|
|       Rockaway Park|    1|       80|
|     Oakland Gardens|    1|       56|
|           Stapleton|    1|        4|
|            Flushing|    1|      250|
|          Ozone Park|    1|      105|
|          Ocean Hill|    1|      354|
|             Maspeth|    1|      290|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|  Brooklyn Navy Yard|    1|       62|
|          Highbridge|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
+--------------------+-----+---------+
only showing top 20 rows
Running with partition_bytes=67108864, 

...

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|                SoHo|    1|    20197|
|Upper East Side S...|    1|   142708|
|       Fordham South|    1|       91|
|             Bayside|    1|       81|
|    Garment District|    1|    48093|
|     Cambria Heights|    1|      143|
|   Kew Gardens Hills|    1|      151|
|    South Ozone Park|    1|      735|
|Breezy Point/Fort...|    1|        6|
|Flushing Meadows-...|    1|      510|
|Riverdale/North R...|    1|       98|
|    Brooklyn Heights|    1|     1257|
|             Bedford|    1|      516|
|        Bloomingdale|    1|     7849|
|            Flatiron|    1|    46496|
|  Morrisania/Melrose|    1|      289|
|       Prospect Park|    1|       53|
|          Douglaston|    1|       49|
|         Fort Greene|    1|     1134|
|            Gramercy|    1|    59367|
+--------------------+-----+---------+
only showing top 20 rows
Run 2 for partition_bytes=67108864, par

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|                SoHo|    1|    20197|
|Upper East Side S...|    1|   142708|
|       Fordham South|    1|       91|
|             Bayside|    1|       81|
|    Garment District|    1|    48093|
|     Cambria Heights|    1|      143|
|   Kew Gardens Hills|    1|      151|
|    South Ozone Park|    1|      735|
|Breezy Point/Fort...|    1|        6|
|Flushing Meadows-...|    1|      510|
|Riverdale/North R...|    1|       98|
|    Brooklyn Heights|    1|     1257|
|             Bedford|    1|      516|
|        Bloomingdale|    1|     7849|
|            Flatiron|    1|    46496|
|  Morrisania/Melrose|    1|      289|
|       Prospect Park|    1|       53|
|          Douglaston|    1|       49|
|         Fort Greene|    1|     1134|
|            Gramercy|    1|    59367|
+--------------------+-----+---------+
only showing top 20 rows
Run 3 for partition_bytes=67108864, par

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|                SoHo|    1|    20197|
|Upper East Side S...|    1|   142708|
|       Fordham South|    1|       91|
|             Bayside|    1|       81|
|    Garment District|    1|    48093|
|     Cambria Heights|    1|      143|
|   Kew Gardens Hills|    1|      151|
|    South Ozone Park|    1|      735|
|Breezy Point/Fort...|    1|        6|
|Flushing Meadows-...|    1|      510|
|Riverdale/North R...|    1|       98|
|    Brooklyn Heights|    1|     1257|
|             Bedford|    1|      516|
|        Bloomingdale|    1|     7849|
|            Flatiron|    1|    46496|
|  Morrisania/Melrose|    1|      289|
|       Prospect Park|    1|       53|
|          Douglaston|    1|       49|
|         Fort Greene|    1|     1134|
|            Gramercy|    1|    59367|
+--------------------+-----+---------+
only showing top 20 rows
Run 4 for partition_bytes=67108864, par

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/bernard/Projects/dic/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bernard/Projects/dic/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bernard/.local/share/uv/python/cpython-3.11.7-linux-x86_64-gnu/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 